# 🗺️ Rainforest Dashboard — Geodata Preparation
Downloads Brazilian state boundaries (IBGE) and RAISG indigenous territories,
simplifies geometries for web performance, and exports as GeoJSON to `webapp/assets/`.

**Run once** before building the Dash app.

In [ ]:
import geopandas as gpd
import requests
import zipfile
import io
import json
import os

# Output directory (relative to notebook location, adjust if running locally)
ASSETS_DIR = "../webapp/assets"
os.makedirs(ASSETS_DIR, exist_ok=True)
os.makedirs("shapefiles", exist_ok=True)

print(f"✓ Output dir: {os.path.abspath(ASSETS_DIR)}")

## Step 1 — Brazilian state boundaries (IBGE)
Official source: IBGE Malhas Territoriais 2022, public domain.

In [ ]:
# Download IBGE state boundaries
url_ibge = (
    "https://geoftp.ibge.gov.br/organizacao_do_territorio/"
    "malhas_territoriais/malhas_municipais/municipio_2022/Brasil/BR/"
    "BR_UF_2022.zip"
)

print("Downloading IBGE state boundaries (~5 MB)...")
resp = requests.get(url_ibge, timeout=180)
resp.raise_for_status()

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    z.extractall("shapefiles/ibge_states/")

print("✓ Downloaded and extracted")

In [ ]:
# Read and simplify
gdf_states = gpd.read_file("shapefiles/ibge_states/")
gdf_states = gdf_states.to_crs("EPSG:4326")

print(f"Columns: {gdf_states.columns.tolist()}")
print(f"States: {len(gdf_states)}")

# Simplify geometry for web performance
gdf_states["geometry"] = gdf_states["geometry"].simplify(0.01, preserve_topology=True)

# Keep only relevant columns (adjust names if IBGE changes schema)
rename = {}
for col in gdf_states.columns:
    if col.upper() in ("SIGLA_UF", "CD_UF"):
        rename[col] = "state_code"
    elif col.upper() in ("NM_UF", "NM_ESTADO"):
        rename[col] = "state_name"

gdf_states = gdf_states.rename(columns=rename)
keep = [c for c in ["state_code", "state_name", "geometry"] if c in gdf_states.columns]
gdf_states = gdf_states[keep]

out_path = os.path.join(ASSETS_DIR, "prodes_states.geojson")
gdf_states.to_file(out_path, driver="GeoJSON")
size_kb = os.path.getsize(out_path) / 1024
print(f"✓ Saved prodes_states.geojson ({size_kb:.0f} KB, {len(gdf_states)} features)")

## Step 2 — RAISG Indigenous Territories
Download from [raisg.org/en/maps/](https://www.raisg.org/en/maps/).

**Manual step if automated download fails:**
1. Go to https://www.raisg.org/en/maps/
2. Download the shapefile for "Protected Areas and Indigenous Territories"
3. Place the .shp (and companion files) at `shapefiles/raisg/`
4. Re-run the cell below

In [ ]:
raisg_shp = "shapefiles/raisg/raisg_territories.shp"
out_path = os.path.join(ASSETS_DIR, "raisg_territories.geojson")

if os.path.exists(raisg_shp):
    gdf_raisg = gpd.read_file(raisg_shp)
    gdf_raisg = gdf_raisg.to_crs("EPSG:4326")
    gdf_raisg["geometry"] = gdf_raisg["geometry"].simplify(0.01, preserve_topology=True)
    gdf_raisg.to_file(out_path, driver="GeoJSON")
    size_kb = os.path.getsize(out_path) / 1024
    print(f"✓ Saved raisg_territories.geojson ({size_kb:.0f} KB, {len(gdf_raisg)} features)")
else:
    # Write empty placeholder so Dash app loads without error
    empty = {"type": "FeatureCollection", "features": []}
    with open(out_path, "w") as f:
        json.dump(empty, f)
    print("⚠️  RAISG shapefile not found — empty placeholder written.")
    print("   Download from https://www.raisg.org/en/maps/ and re-run.")

## ✅ Done
GeoJSON files are ready in `webapp/assets/`:
- `prodes_states.geojson` — Brazilian state boundaries for choropleth
- `raisg_territories.geojson` — Indigenous territories overlay (or empty placeholder)

Copy both files into the Dash container's `assets/` directory before building Docker.